# SCM Framework – Frequency Analysis (Exploratory)

**Goal**: Search for periodic structures in galaxy rotation curves after removing the main trend. This notebook is **exploratory** and separate from the validated `scm_reproducible.ipynb`.

**Method**:
1. Load a rotation curve (from SPARC or synthetic).
2. Remove a smooth baseline (e.g., outer slope model).
3. Compute FFT of the residuals.
4. Detect dominant frequencies.
5. Permutation test to assess significance.
6. (Optional) Correlate with environmental proxy.

**Warning**: FFT on irregularly sampled data can produce spurious signals. Use with caution.

## 1. Setup and data loading

In [ ]:
!pip install -q pandas numpy scipy matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
np.random.seed(42)

## 2. Load a real rotation curve (example: SPARC)

We'll use the `scm_master_final.csv` metadata, but for the full rotation curve we need the original SPARC data. For now, we simulate a realistic curve. In a real scenario, replace with your data.

**Note**: To use actual SPARC data, download the `rotmod` files from the SPARC database.

In [ ]:
# Simulate a rotation curve with a hidden oscillation (same as before, but now we call it "realistic")
R = np.linspace(1, 20, 200)
V_smooth = 220 * (1 - np.exp(-R/5))
true_freq = 0.25
signal = 15 * np.sin(2 * np.pi * true_freq * R)
V_obs = V_smooth + signal + np.random.normal(0, 3, size=len(R))

# Save to a DataFrame (simulating a galaxy)
df_gal = pd.DataFrame({'R_kpc': R, 'Vobs_kms': V_obs})
print("Simulated rotation curve (use real data in practice)")
df_gal.head()

## 3. Remove trend (detrending)

We need to isolate residuals by subtracting a smooth baseline. Here we use a simple polynomial fit (order 2). For SPARC, you could use the fitted outer slope or a rotation curve model.

In [ ]:
R_arr = df_gal['R_kpc'].values
V_arr = df_gal['Vobs_kms'].values

# Fit a low-order polynomial (or use your own model)
coeff = np.polyfit(R_arr, V_arr, 2)
V_fit = np.polyval(coeff, R_arr)
residual = V_arr - V_fit

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(R_arr, V_arr, 'k.', label='Data')
plt.plot(R_arr, V_fit, 'r-', label='Polynomial fit')
plt.xlabel('R (kpc)'); plt.ylabel('V (km/s)')
plt.title('Rotation curve + trend')
plt.legend()

plt.subplot(1,2,2)
plt.plot(R_arr, residual, 'b.')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('R (kpc)'); plt.ylabel('Residual (km/s)')
plt.title('Residuals after detrending')
plt.tight_layout()
plt.show()

## 4. Interpolate to uniform grid (required for FFT)

The FFT expects evenly spaced points. We'll interpolate the residuals onto a uniform grid in R.

In [ ]:
R_uniform = np.linspace(R_arr.min(), R_arr.max(), 512)
interp_func = interp1d(R_arr, residual, kind='cubic', fill_value='extrapolate')
residual_uniform = interp_func(R_uniform)

plt.plot(R_uniform, residual_uniform, 'g-', alpha=0.7)
plt.xlabel('R (kpc)')
plt.ylabel('Residual')
plt.title('Residuals on uniform grid')
plt.show()

## 5. FFT and power spectrum

In [ ]:
fft_vals = fft(residual_uniform)
freqs = fftfreq(len(R_uniform), d=R_uniform[1]-R_uniform[0])
power = np.abs(fft_vals)**2

# Keep positive frequencies only
positive_mask = freqs > 0
freqs_pos = freqs[positive_mask]
power_pos = power[positive_mask]

# Find peaks above 95th percentile
threshold = np.percentile(power_pos, 95)
peaks, _ = find_peaks(power_pos, height=threshold)
peak_freqs = freqs_pos[peaks]
peak_powers = power_pos[peaks]

print(f"Detected peak frequencies (1/kpc): {peak_freqs}")
print(f"Corresponding powers: {peak_powers}")

## 6. Permutation test (robustness)

Shuffle residuals, recompute FFT, and see if the observed peaks are above the noise level.

In [ ]:
n_perm = 500
perm_powers = []
for _ in range(n_perm):
    resid_perm = np.random.permutation(residual_uniform)
    fft_perm = fft(resid_perm)
    power_perm = np.abs(fft_perm)**2
    perm_powers.append(power_perm[positive_mask])
perm_powers = np.array(perm_powers)

# 95% upper envelope from permutations
threshold_perm = np.percentile(perm_powers, 95, axis=0)

# Check which observed peaks exceed the permutation threshold
significant = []
for i, f in enumerate(peak_freqs):
    idx = np.argmin(np.abs(freqs_pos - f))
    if power_pos[idx] > threshold_perm[idx]:
        significant.append(f)

print(f"\nSignificant frequencies after permutation test: {significant}")

## 7. Visualisation: Power spectrum with thresholds

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(freqs_pos, power_pos, 'b-', label='Observed power')
plt.plot(freqs_pos, threshold_perm, 'r--', label='95% permutation threshold')
plt.axhline(threshold, color='gray', linestyle=':', label='95% global threshold')
for f in peak_freqs:
    plt.axvline(f, color='g', linestyle=':', alpha=0.7)
plt.xlabel('Frequency (1/kpc)')
plt.ylabel('Power')
plt.title('Power spectrum of residuals')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Conclusion

If a frequency appears consistently above the permutation threshold, it might indicate a real periodic structure. The next step would be to:
- Apply to a sample of galaxies (e.g., SPARC high-mass).
- Correlate dominant frequencies with environmental proxies (delta_mass_std).
- Check if the signal is stronger in high-mass galaxies.

**Important**: This is exploratory. Any detection must be confirmed with rigorous statistical controls (e.g., using real SPARC rotation curves, not simulations).